# Pokemon TCG AI Battle — Agent Testing

Run inside a Kaggle competition notebook (cabt is pre-installed there).

1. Set `GITHUB_URL`.
2. Edit `AGENTS` and `N_GAMES`.
3. Run all cells.

## 1. Clone repo

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"  # replace this
!git clone {GITHUB_URL} /kaggle/working/repo

In [ ]:
import os, sys
os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo/src")

## 2. Card database

Loads card and attack metadata from the C library.  
**If this fails, `RuleBasedAgent` is identical to `RandomAgent`** — every rule
that needs card data silently falls back to `RandomFallback`.

In [ ]:
from ptcg import card_db

loaded = card_db.load()
print(f"card_db loaded : {loaded}")

if loaded:
    print(f"Cards          : {len(card_db._cards)}")
    print(f"Attacks        : {len(card_db._attacks)}")
    sample = card_db.get_card(721)
    if sample:
        print(f"Sample (721)   : {sample.get('name')}  "
              f"HP={sample.get('hp')}  "
              f"type={sample.get('energyType')}  "
              f"weakness={sample.get('weakness')}")
else:
    print()
    print("WARNING: card_db failed to load.")
    print("RuleBasedAgent will behave identically to RandomAgent.")

## 3. Available deck

In [ ]:
from collections import Counter
from kaggle_environments.envs.cabt.cabt import deck as DEFAULT_DECK

counts = Counter(DEFAULT_DECK)
print(f"Default deck — {len(DEFAULT_DECK)} cards\n")
print(f"{'Card ID':<10} {'Name':<30} {'Count'}")
print("-" * 50)
for card_id, count in sorted(counts.items()):
    name = card_db.get_card(card_id).get("name", "?") if loaded else "?"
    print(f"{card_id:<10} {name:<30} {count}")

## 4. Configure tournament

In [ ]:
AGENTS  = [
    "RandomAgent",
    "RuleBasedAgent",
]

N_GAMES = 20  # games per matchup — use >= 20 for a reliable win-rate estimate

## 5. Run tournament

In [ ]:
import importlib, re
from collections import defaultdict
from kaggle_environments import make


def load_agent(class_name: str):
    module_name = re.sub(r"(?<!^)(?=[A-Z])", "_", class_name).lower()
    module = importlib.import_module(f"ptcg.agents.{module_name}")
    return getattr(module, class_name)()


def run_battle(agent0, agent1):
    env = make("cabt", configuration={"decks": [agent0.get_deck(), agent1.get_deck()]})
    env.run([agent0, agent1])
    rewards = [step["reward"] for step in env.steps[-1]]
    return rewards, env


records  = defaultdict(lambda: {"wins": 0, "draws": 0, "losses": 0})
last_env = None

for name0 in AGENTS:
    for name1 in AGENTS:
        key = (name0, name1)
        for _ in range(N_GAMES):
            a0, a1 = load_agent(name0), load_agent(name1)
            a0.on_game_start()
            a1.on_game_start()
            rewards, last_env = run_battle(a0, a1)
            result = {"steps": last_env.steps, "rewards": rewards}
            a0.on_game_end(result)
            a1.on_game_end(result)
            r0 = rewards[0]
            if   r0 == 1:  records[key]["wins"]   += 1
            elif r0 == 0:  records[key]["draws"]  += 1
            else:          records[key]["losses"] += 1

        w, d, l = records[key]["wins"], records[key]["draws"], records[key]["losses"]
        print(f"{name0} vs {name1:<20}  {w}W {d}D {l}L  ({w/N_GAMES*100:.0f}% win rate for {name0})")

## 6. Results summary

In [ ]:
print(f"{'Agent 0':<20} {'Agent 1':<20} {'W':>4} {'D':>4} {'L':>4} {'Win%':>6}")
print("-" * 65)
for (name0, name1), r in records.items():
    w, d, l  = r["wins"], r["draws"], r["losses"]
    total    = w + d + l
    rate     = w / total * 100 if total > 0 else 0
    print(f"{name0:<20} {name1:<20} {w:>4} {d:>4} {l:>4} {rate:>5.0f}%")

## 7. Game replay (last battle)

Human-readable replay of the last game played, using card and attack names
from `card_db`. Noisy bookkeeping events (shuffles, draws) are suppressed.

In [ ]:
from ptcg.log_formatter import format_history

history = last_env.steps[0][0].get("visualize")

if history is None:
    print("No visualize data available.")
else:
    print(format_history(history))